In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import pandas as pd
import numpy as np

from config.paths import RAW_SALES_FILE

from utils.file_utils import read_csv

In [3]:
sales_data = read_csv(RAW_SALES_FILE)
sales_data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,01-12-2010 08:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,01-12-2010 08:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,01-12-2010 08:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,01-12-2010 08:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,01-12-2010 08:26,3.39,17850.0,United Kingdom


In [4]:
# data structure & schema
print(f"Rows    : {sales_data.shape[0]:,}")
print(f"Columns : {sales_data.shape[1]}")

Rows    : 541,909
Columns : 8


In [9]:
sales_data["Description"].nunique()

4223

In [10]:
sales_data["Country"].nunique()

38

In [11]:
sales_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [12]:
# datatime 

sales_data["InvoiceDate"] = pd.to_datetime(
    sales_data["InvoiceDate"],
    format="%d-%m-%Y %H:%M"
)

In [13]:
# Does CustomerID contain decimal values?
decimal_ids = sales_data.loc[
    sales_data["CustomerID"].notna() &
    (sales_data["CustomerID"] % 1 != 0),
    "CustomerID"
]

print(f"Customer IDs with decimal values : {len(decimal_ids)}")

Customer IDs with decimal values : 0


In [14]:
sales_data["CustomerID"] = sales_data["CustomerID"].astype("Int64")

In [61]:
sales_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  Int64         
 7   Country      541909 non-null  object        
dtypes: Int64(1), datetime64[ns](1), float64(1), int64(1), object(4)
memory usage: 33.6+ MB


In [62]:
# Country Standardization
print(f"Unique Countries : {sales_data['Country'].nunique()}")

sorted(sales_data["Country"].unique())

Unique Countries : 38


['Australia',
 'Austria',
 'Bahrain',
 'Belgium',
 'Brazil',
 'Canada',
 'Channel Islands',
 'Cyprus',
 'Czech Republic',
 'Denmark',
 'EIRE',
 'European Community',
 'Finland',
 'France',
 'Germany',
 'Greece',
 'Hong Kong',
 'Iceland',
 'Israel',
 'Italy',
 'Japan',
 'Lebanon',
 'Lithuania',
 'Malta',
 'Netherlands',
 'Norway',
 'Poland',
 'Portugal',
 'RSA',
 'Saudi Arabia',
 'Singapore',
 'Spain',
 'Sweden',
 'Switzerland',
 'USA',
 'United Arab Emirates',
 'United Kingdom',
 'Unspecified']

In [63]:
special_locations = [
    "EIRE",
    "RSA",
    "European Community",
    "Unspecified"
]

sales_data.loc[
    sales_data["Country"].isin(special_locations),
    "Country"
].value_counts()

Country
EIRE                  8196
Unspecified            446
European Community      61
RSA                     58
Name: count, dtype: int64

In [64]:
unspecified = (
    sales_data.loc[
        sales_data["Country"] == "Unspecified"
    ]
)

print(f"Transactions : {len(unspecified):,}")
print(f"Revenue      : {(unspecified['Quantity'] * unspecified['UnitPrice']).sum():,.2f}")

unspecified.head()

Transactions : 446
Revenue      : 4,749.79


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
152712,549687,20685,DOORMAT RED RETROSPOT,2,2011-04-11 13:29:00,7.95,12363,Unspecified
152713,549687,22691,DOORMAT WELCOME SUNRISE,2,2011-04-11 13:29:00,7.95,12363,Unspecified
152714,549687,48116,DOORMAT MULTICOLOUR STRIPE,2,2011-04-11 13:29:00,7.95,12363,Unspecified
152715,549687,21213,PACK OF 72 SKULL CAKE CASES,24,2011-04-11 13:29:00,0.55,12363,Unspecified
152716,549687,21977,PACK OF 60 PINK PAISLEY CAKE CASES,24,2011-04-11 13:29:00,0.55,12363,Unspecified


In [65]:
unspecified["CustomerID"].isna().value_counts()

CustomerID
False    244
True     202
Name: count, dtype: int64

In [66]:
unspecified["InvoiceNo"].astype(str).str.startswith("C").value_counts()

InvoiceNo
False    446
Name: count, dtype: int64

EIRE, RSA and European Community  are business abbreviation, Unspecified" represents an unknown customer location, not an invalid transaction.

The transactions:

are genuine sales,
generate revenue,
include identifiable customers,
are not cancellations.

In [67]:
missing_values = (
    sales_data
    .isna()
    .sum()
    .to_frame("Missing Values")
)

missing_values["Missing (%)"] = (
    missing_values["Missing Values"]
    / len(sales_data)
    * 100
).round(2)

missing_values

,Missing Values,Missing (%)
InvoiceNo,0,0.00
StockCode,0,0.00
Description,1454,0.27
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
CustomerID,135080,24.93
Country,0,0.00


In [68]:
# inspection of missing description
missing_description = (
    sales_data.loc[
        sales_data["Description"].isna()
    ]
)

print(f"Total Records : {len(missing_description):,}")

missing_description.head(10)

Total Records : 1,454


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,<NA>,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,<NA>,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,<NA>,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,<NA>,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,<NA>,United Kingdom
1988,536550,85044,NaN,1,2010-12-01 14:34:00,0.0,<NA>,United Kingdom
2024,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,<NA>,United Kingdom
2025,536553,37461,NaN,3,2010-12-01 14:35:00,0.0,<NA>,United Kingdom
2026,536554,84670,NaN,23,2010-12-01 14:35:00,0.0,<NA>,United Kingdom
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,<NA>,United Kingdom


In [69]:
# Are all missing descriptions associated with the same StockCode?
print(f"Unique StockCodes : {missing_description['StockCode'].nunique():,}")

(
    missing_description["StockCode"]
    .value_counts()
    .head(20)
)

Unique StockCodes : 960


StockCode
23084    10
35965    10
22084     9
22451     6
22501     5
21067     5
23348     5
21784     5
22866     5
21033     5
79321     4
84944     4
22720     4
POST      4
22502     4
79164     4
20713     4
20975     4
82583     4
21429     4
Name: count, dtype: int64

In [70]:
# Can descriptions be recovered
stockcodes = missing_description["StockCode"].unique()

(
    sales_data.loc[
        sales_data["StockCode"].isin(stockcodes),
        ["StockCode", "Description"]
    ]
    .drop_duplicates()
    .sort_values("StockCode")
)

,StockCode,Description
139083,10002,NaN
31,10002,INFLATABLE POLITICAL GLOBE
279310,10080,NaN
103332,10080,GROOVY CACTUS INFLATABLE
454350,10080,check
...,...,...
52262,POST,NaN
317501,gift_0001_10,NaN
112442,gift_0001_10,Dotcomgiftshop Gift Voucher £10.00
44725,gift_0001_30,Dotcomgiftshop Gift Voucher £30.00


In [71]:
# Are these genuine sales?
missing_description[
    [
        "Quantity",
        "UnitPrice"
    ]
].describe()

,Quantity,UnitPrice
count,1454.000000,1454.0
mean,-9.359697,0.0
std,243.238758,0.0
min,-3667.000000,0.0
25%,-24.000000,0.0
50%,-3.000000,0.0
75%,4.000000,0.0
max,5568.000000,0.0


In [72]:
# drop the missing product 

missing description is 99 belong to same stockcode out of 1456, but they have 0 unit price and they don't belongs to cancelled order or return, they can be returned using the stockcode but they don't share the contribuition on revenue.

In [73]:
# Remove records with missing product descriptions
sales_data = (
    sales_data
    .dropna(subset=["Description"])
    .reset_index(drop=True)
)

In [74]:
print(f"Rows after removal : {len(sales_data):,}")
print(f"Missing Descriptions : {sales_data['Description'].isna().sum()}")

Rows after removal : 540,455
Missing Descriptions : 0


### Missing CustomerID

In [75]:
missing_customer = (
    sales_data.loc[
        sales_data["CustomerID"].isna()
    ]
)

print(f"Missing CustomerID Records : {len(missing_customer):,}")

Missing CustomerID Records : 133,626


In [76]:
# Which countries do they belong to
(
    missing_customer["Country"]
    .value_counts()
    .to_frame(name="Transactions")
)

,Transactions
Country,
United Kingdom,132146
EIRE,711
Hong Kong,288
Unspecified,202
Switzerland,125
France,66
Israel,47
Portugal,39
Bahrain,2


In [77]:
# Are they concentrated in one country?
country_distribution = (
    missing_customer["Country"]
    .value_counts()
    .to_frame(name="Transactions")
)

country_distribution["Percentage"] = (
    country_distribution["Transactions"]
    / len(missing_customer)
    * 100
).round(2)

country_distribution

,Transactions,Percentage
Country,,
United Kingdom,132146,98.89
EIRE,711,0.53
Hong Kong,288,0.22
Unspecified,202,0.15
Switzerland,125,0.09
France,66,0.05
Israel,47,0.04
Portugal,39,0.03
Bahrain,2,0.00


In [78]:
# Do they represent a large share of revenue?
missing_customer = missing_customer.copy()

missing_customer["Revenue"] = (
    missing_customer["Quantity"]
    * missing_customer["UnitPrice"]
)
missing_customer["Revenue"].describe()

count    133626.000000
mean         10.833836
std         158.821030
min      -17836.460000
25%           2.490000
50%           4.960000
75%          10.790000
max       13541.330000
Name: Revenue, dtype: float64

In [79]:
total_revenue = (
    sales_data["Quantity"]
    * sales_data["UnitPrice"]
).sum()

missing_revenue = missing_customer["Revenue"].sum()

print(f"Revenue from Missing CustomerID Records : {missing_revenue:,.2f}")
print(f"Total Revenue                         : {total_revenue:,.2f}")
print(f"Revenue Share                         : {(missing_revenue / total_revenue) * 100:.2f}%")

Revenue from Missing CustomerID Records : 1,447,682.12
Total Revenue                         : 9,747,747.93
Revenue Share                         : 14.85%


In [80]:
# Are they mainly cancelled invoices?
missing_customer["IsCancelled"] = (
    missing_customer["InvoiceNo"]
    .astype(str)
    .str.startswith("C")
)

missing_customer["IsCancelled"].value_counts()

IsCancelled
False    133243
True        383
Name: count, dtype: int64

In [81]:
# Are they associated with specific products?
(
    missing_customer["Description"]
    .value_counts()
    .head(20)
)

Description
DOTCOM POSTAGE                        693
JUMBO BAG RED RETROSPOT               497
JUMBO STORAGE BAG SUKI                414
JUMBO SHOPPER VINTAGE RED PAISLEY     388
JUMBO BAG WOODLAND ANIMALS            372
JUMBO BAG PINK POLKADOT               348
RECYCLING BAG RETROSPOT               341
RED TOADSTOOL LED NIGHT LIGHT         328
SUKI  SHOULDER BAG                    326
GREEN REGENCY TEACUP AND SAUCER       324
PARTY BUNTING                         311
PACK OF 72 RETROSPOT CAKE CASES       305
WOODLAND CHARLOTTE BAG                304
RED RETROSPOT CHARLOTTE BAG           301
CHARLOTTE BAG SUKI DESIGN             300
WHITE HANGING HEART T-LIGHT HOLDER    299
REGENCY CAKESTAND 3 TIER              295
JAM MAKING SET PRINTED                294
RECIPE BOX PANTRY YELLOW DESIGN       290
JUMBO  BAG BAROQUE BLACK WHITE        287
Name: count, dtype: int64

In [82]:
# Most common StockCodes.
(
    missing_customer["StockCode"]
    .value_counts()
    .head(20)
)

StockCode
DOT       693
85099B    497
21931     414
22411     388
20712     372
22197     358
22386     348
22379     341
21731     328
21935     326
22697     324
47566     311
21212     305
20719     304
22355     301
20724     301
22423     298
22961     294
22666     290
85099C    287
Name: count, dtype: int64

Missing CustomerIDs are not a global data quality issue. They are almost entirely associated with transactions from the United Kingdom. This strongly suggests a business process rather than missing data.

Note: Missing CustomerID transactions involve real products and genuine sales. So keep the records.

### Duplicate

In [83]:
duplicate_count = sales_data.duplicated().sum()

print(f"Duplicate Records : {duplicate_count:,}")

duplicate_percentage = (
    duplicate_count
    / len(sales_data)
    * 100
)

print(f"Percentage : {duplicate_percentage:.2f}%")

Duplicate Records : 5,268
Percentage : 0.97%


In [84]:
# Are they truly identical across all columns?
duplicate_records = sales_data.loc[
    sales_data.duplicated(keep=False)
].sort_values(
    by=["InvoiceNo", "StockCode"]
)

print(f"Duplicate Records : {len(duplicate_records):,}")

duplicate_records.head(20)

Duplicate Records : 10,147


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908,United Kingdom
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908,United Kingdom
565,536412,21448,12 DAISY PEGS IN WOOD BOX,2,2010-12-01 11:49:00,1.65,17920,United Kingdom
578,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,17920,United Kingdom


Note: They are truely identical across all the columns so removing it.

In [85]:
# Remove exact duplicate records
sales_data = (
    sales_data
    .drop_duplicates()
    .reset_index(drop=True)
)

In [86]:
duplicate_count = sales_data.duplicated().sum()

print(f"Remaining Duplicate Records : {duplicate_count:,}")
print(f"Total Records : {len(sales_data):,}")

Remaining Duplicate Records : 0
Total Records : 535,187


### Negatives Values

In [87]:
negative_quantity = sales_data.loc[
    sales_data["Quantity"] < 0
].copy()

print(f"Negative Quantity Records : {len(negative_quantity):,}")

Negative Quantity Records : 9,725


In [88]:
Are they cancellation invoices?
negative_quantity["IsCancelled"] = (
    negative_quantity["InvoiceNo"]
    .astype(str)
    .str.startswith("C")
)

negative_quantity["IsCancelled"].value_counts()

Object `invoices` not found.


IsCancelled
True     9251
False     474
Name: count, dtype: int64

In [89]:
negative_without_cancel = (
    negative_quantity.loc[
        ~negative_quantity["IsCancelled"]
    ]
)

print(f"Records : {len(negative_without_cancel):,}")

negative_without_cancel.head(20)

Records : 474


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsCancelled
7169,537032,21275,?,-30,2010-12-03 16:50:00,0.0,<NA>,United Kingdom,False
12917,537425,84968F,check,-20,2010-12-06 15:35:00,0.0,<NA>,United Kingdom,False
12918,537426,84968E,check,-35,2010-12-06 15:36:00,0.0,<NA>,United Kingdom,False
12962,537432,35833G,damages,-43,2010-12-06 16:10:00,0.0,<NA>,United Kingdom,False
20935,538072,22423,faulty,-13,2010-12-09 14:10:00,0.0,<NA>,United Kingdom,False
21114,538090,20956,?,-723,2010-12-09 14:48:00,0.0,<NA>,United Kingdom,False
21870,538161,46000S,Dotcom sales,-100,2010-12-09 17:25:00,0.0,<NA>,United Kingdom,False
21871,538162,46000M,Dotcom sales,-100,2010-12-09 17:25:00,0.0,<NA>,United Kingdom,False
41929,540010,22501,reverse 21/5/10 adjustment,-100,2011-01-04 11:13:00,0.0,<NA>,United Kingdom,False
41930,540012,22502,reverse 21/5/10 adjustment,-100,2011-01-04 11:14:00,0.0,<NA>,United Kingdom,False


A detailed investigation identified 474 negative-quantity records that were not associated with cancellation invoices. These transactions contained operational descriptions such as "damages", "faulty", "counted", and "label mix up", had a unit price of zero, generated no revenue, and were not linked to identifiable customers. The records represent internal inventory and administrative adjustments rather than genuine customer sales or returns. Consequently, these records were excluded from the analytical dataset.

In [90]:
# Remove internal inventory adjustment records
sales_data = (
    sales_data.loc[
        ~(
            (sales_data["Quantity"] < 0) &
            (~sales_data["InvoiceNo"].astype(str).str.startswith("C"))
        )
    ]
    .reset_index(drop=True)
)

In [91]:
# Validate removal of internal inventory adjustment records
remaining_invalid = sales_data.loc[
    (sales_data["Quantity"] < 0) &
    (~sales_data["InvoiceNo"].astype(str).str.startswith("C"))
]

print(f"Remaining Invalid Negative Quantity Records : {len(remaining_invalid):,}")

remaining_invalid.head()

Remaining Invalid Negative Quantity Records : 0


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country


### Negative Price

In [92]:
negative_price = sales_data.loc[
    sales_data["UnitPrice"] < 0
]

zero_price = sales_data.loc[
    sales_data["UnitPrice"] == 0
]

print(f"Negative Prices : {len(negative_price):,}")
print(f"Zero Prices     : {len(zero_price):,}")

Negative Prices : 2
Zero Prices     : 582


In [93]:
negative_price

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
296344,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,<NA>,United Kingdom
296345,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,<NA>,United Kingdom


In [94]:
# Remove records with negative unit prices
sales_data = (
    sales_data.loc[
        sales_data["UnitPrice"] >= 0
    ]
    .reset_index(drop=True)
)

Note: Two records with negative unit prices were identified during the data quality assessment. Manual inspection confirmed that both represented accounting adjustments ("Adjust bad debt") rather than customer sales or product returns. The records contained negative unit prices, missing customer identifiers, and non-standard invoice numbers prefixed with "A". As these transactions do not represent revenue-generating retail activity and would distort sales performance metrics, they were excluded from the analytical dataset.

In [95]:
# prices
zero_price["IsCancelled"] = (
    zero_price["InvoiceNo"]
    .astype(str)
    .str.startswith("C")
)

zero_price["IsCancelled"].value_counts()

IsCancelled
False    582
Name: count, dtype: int64

In [96]:
(zero_price["Quantity"] > 0).value_counts()

Quantity
True    582
Name: count, dtype: int64

In [97]:
zero_price["Description"].value_counts().head(30)

Description
check                                39
found                                25
adjustment                           14
FRENCH BLUE METAL DOOR SIGN 1         9
FRENCH BLUE METAL DOOR SIGN 8         8
amazon                                8
Found                                 8
FRENCH BLUE METAL DOOR SIGN 3         7
RECIPE BOX PANTRY YELLOW DESIGN       7
OWL DOORSTOP                          7
FRENCH BLUE METAL DOOR SIGN 4         7
Amazon                                7
FRENCH BLUE METAL DOOR SIGN No        7
RED KITCHEN SCALES                    6
?                                     6
FRENCH BLUE METAL DOOR SIGN 6         6
FRENCH BLUE METAL DOOR SIGN 5         6
FRENCH BLUE METAL DOOR SIGN 7         6
Manual                                6
FRENCH BLUE METAL DOOR SIGN 2         5
FRENCH BLUE METAL DOOR SIGN 9         5
RED RETROSPOT CHARLOTTE BAG           5
MINT KITCHEN SCALES                   5
RECIPE BOX BLUE SKETCHBOOK DESIGN     5
POLYESTER FILLER PAD 45x45cm

In [98]:
zero_price["CustomerID"].isna().value_counts()

CustomerID
True     542
False     40
Name: count, dtype: int64

In [99]:
zero_price["InvoiceNo"].nunique()

227

In [100]:
# Operational descriptions that do not represent customer sales
operational_descriptions = [
    "check",
    "found",
    "Found",
    "adjustment",
    "Manual",
    "amazon",
    "Amazon",
    "?",
    "had been put aside"
]

# Remove operational records with zero unit price
sales_data = (
    sales_data.loc[
        ~(
            (sales_data["UnitPrice"] == 0) &
            (sales_data["Description"].isin(operational_descriptions))
        )
    ]
    .reset_index(drop=True)
)

In [101]:
# Verify that no operational zero-price records remain
remaining_operational = sales_data.loc[
    (sales_data["UnitPrice"] == 0) &
    (sales_data["Description"].isin(operational_descriptions))
]

print(f"Remaining Operational Zero-Price Records : {len(remaining_operational):,}")

Remaining Operational Zero-Price Records : 0


In [102]:
remaining_zero_price = sales_data.loc[
    sales_data["UnitPrice"] == 0
]

print(f"Remaining Zero-Price Records : {len(remaining_zero_price):,}")

remaining_zero_price["Description"].value_counts().head(20)

Remaining Zero-Price Records : 464


Description
FRENCH BLUE METAL DOOR SIGN 1        9
FRENCH BLUE METAL DOOR SIGN 8        8
FRENCH BLUE METAL DOOR SIGN 4        7
FRENCH BLUE METAL DOOR SIGN No       7
OWL DOORSTOP                         7
RECIPE BOX PANTRY YELLOW DESIGN      7
FRENCH BLUE METAL DOOR SIGN 3        7
FRENCH BLUE METAL DOOR SIGN 6        6
FRENCH BLUE METAL DOOR SIGN 5        6
RED KITCHEN SCALES                   6
FRENCH BLUE METAL DOOR SIGN 7        6
CHILDS GARDEN SPADE BLUE             5
BOX OF 24 COCKTAIL PARASOLS          5
RECIPE BOX BLUE SKETCHBOOK DESIGN    5
FRENCH BLUE METAL DOOR SIGN 2        5
FRENCH BLUE METAL DOOR SIGN 0        5
FRENCH BLUE METAL DOOR SIGN 9        5
RED RETROSPOT CHARLOTTE BAG          5
IVORY KITCHEN SCALES                 5
DOORMAT WELCOME TO OUR HOME          5
Name: count, dtype: int64

Among the 582 zero-price transactions, only records representing internal operational activities 40 records (e.g., check, found, adjustment, Manual, and Amazon) were removed. The remaining zero-price transactions were retained, as they represent legitimate customer product transactions.

## Data Quality Findings

The initial assessment identified several data quality issues that should be addressed before proceeding with data cleaning and business analysis. Although the dataset is structurally complete, several quality concerns may affect the accuracy of revenue, customer, product, and operational reporting if left unresolved.

| Issue                        | Key Finding                                                                                                                                                                                                     | Action                                                                          |
| ---------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------- |
| Invoice Date Stored as Text  | `InvoiceDate` is stored as a text field, preventing time-based analysis.                                                                                                                                        | Convert to `datetime`.                                                          |
| CustomerID Data Type         | Customer identifiers contain no invalid decimal values. The `float64` type is caused by missing values rather than incorrect data.                                                                              | Convert to nullable `Int64` during cleaning.                                    |
| Missing Product Descriptions | **1,454** records (**0.27%**) have missing descriptions. Investigation confirmed all had **zero unit price** and did not contribute to revenue.                                                                 | Remove from the analytical dataset.                                             |
| Missing CustomerID           | **135,080** transactions (**24.93%**) have missing CustomerIDs. **98.9%** originate from the United Kingdom and contribute **14.85%** of total revenue.                                                         | Retain for revenue analysis; exclude from customer-level analysis.              |
| Duplicate Records            | **5,268** duplicate records (**0.97%**) were identified across **1,933** invoices. Duplicate rows were identical across all transaction attributes.                                                             | Remove duplicate records.                                                       |
| Negative Quantity            | **474** non-cancellation records were identified as internal inventory and administrative adjustments with **zero unit price** and no revenue impact.                                                           | Remove operational adjustment records; retain legitimate customer returns.      |
| Negative Unit Price          | Only **2** records contained negative unit prices. Manual inspection confirmed they were **bad debt accounting adjustments**, not retail sales.                                                                 | Remove from the analytical dataset.                                             |
| Zero Unit Price              | **582** zero-price transactions were identified. Only operational records (e.g., **check**, **found**, **adjustment**, **Manual**, **amazon**) were excluded, while genuine product transactions were retained. | Remove operational records; retain legitimate zero-price customer transactions. |


In [103]:
sales_data.shape

(534593, 8)